<h3>문자, 재귀, 토큰 청킹</h3>
<p><strong>개념</strong> 청킹은 긴 문서를 검색 단위로 나누는 작업입니다. 같은 크기 숫자라도 문자를 세는지 토큰을 세는지에 따라 결과가 달라집니다.</p>

In [1]:
# %pip은 현재 커널에 청킹과 토큰 계산 패키지를 설치합니다. -q는 설치 로그를 줄입니다.
# 버전 고정은 같은 입력에서 같은 분할 API를 사용하기 위한 준비입니다.
%pip install -q "langchain-text-splitters==1.1.2" "tiktoken==0.14.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# RecursiveCharacterTextSplitter는 여러 구분자를 순서대로 적용해 텍스트를 나눕니다.
# text는 세 가지 청킹 방식에 공통으로 전달할 한국어 원문입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.

환불은 수령 후 7일 이내 신청합니다."""


<h3>같은 문서를 세 기준으로 나누기</h3>
<p><strong>확인할 점</strong> 문자 방식은 고정 폭과 중복을 직접 적용합니다. 재귀 방식은 빈 줄과 문장 경계를 먼저 사용하고, 토큰 방식은 모델 입력 한도를 관리합니다.</p>

In [3]:
# character_chunks()는 size만큼 자르고 overlap만큼 앞부분을 다시 포함한 리스트를 반환합니다.
# overlap_length()는 두 청크의 접미사와 접두사가 같은 최대 문자 수를 계산합니다.
def character_chunks(value: str, size: int = 35, overlap: int = 8) -> list[str]:
    step = size - overlap
    return [value[i:i + size] for i in range(0, len(value), step)]
def overlap_length(left: str, right: str) -> int:
    limit = min(len(left), len(right))
    return max((n for n in range(1, limit + 1) if left[-n:] == right[:n]), default=0)
character = character_chunks(text)
print("character:", character)
print("character overlaps:", [overlap_length(a, b) for a, b in zip(character, character[1:])])


character: ['배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니', '틀이 더 걸립니다.\n\n환불은 수령 후 7일 이내 신청합니다.', '신청합니다.']
character overlaps: [8, 6]


In [4]:
# 재귀 방식은 문장 구분자를 우선하고, 토큰 방식은 cl100k_base 토큰 수를 기준으로 나눕니다.
# split_text()의 반환값은 순서를 유지한 문자열 청크 리스트입니다.
recursive = RecursiveCharacterTextSplitter(
    chunk_size=35, chunk_overlap=8, keep_separator="end",
    separators=["\n\n", ". ", " ", ""]
).split_text(text)
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=20, chunk_overlap=4
)
token_chunks = token_splitter.split_text(text)
print("recursive:", recursive)
print("recursive overlaps:", [overlap_length(a, b) for a, b in zip(recursive, recursive[1:])])
print("token chunks:", token_chunks)


recursive: ['배송은 결제 다음 날 시작합니다.', '제주 지역은 이틀이 더 걸립니다.', '환불은 수령 후 7일 이내 신청합니다.']
recursive overlaps: [0, 0]
token chunks: ['배송은 결제 다음 날 시작합니다. 제주 지역은', '지역은 이틀이 더 걸립니다.', '환불은 수령 후 7일 이내 신청합니다.']


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 오버랩이 청크 크기 이상이면 다음 시작점이 앞으로 나아가지 않습니다. 설정 오류를 실행 전에 차단합니다.</p>

In [5]:
# 오버랩이 청크 크기와 같아 시작점이 전진하지 않는 설정을 재현합니다.
# assert 조건이 거짓이면 AssertionError가 발생하며 해당 오류만 포착합니다.
try:
    chunk_size, overlap = 20, 20
    assert 0 <= overlap < chunk_size, "overlap은 chunk_size보다 작아야 합니다"
except AssertionError as error:
    print(f"caught: {error}")


caught: overlap은 chunk_size보다 작아야 합니다


<p><strong>수정 방법</strong> 오버랩을 청크 크기보다 작게 설정한 뒤, 같은 문자 분할 함수로 청크 길이와 개수를 검증합니다.</p>

In [6]:
# 오버랩을 청크 크기보다 작게 고치고 같은 문자 분할 함수를 다시 호출합니다.
# all()은 반환된 모든 청크가 최대 길이 조건을 만족하는지 검증합니다.
chunk_size, overlap = 20, 5
assert 0 <= overlap < chunk_size
fixed = character_chunks(text, chunk_size, overlap)
assert all(len(chunk) <= chunk_size for chunk in fixed)
print({"chunk_size": chunk_size, "overlap": overlap, "chunks": len(fixed)})


{'chunk_size': 20, 'overlap': 5, 'chunks': 4}
